In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__notebook__.ipynb
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__output__.json
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/__results__.html
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/custom.css
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_train.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/X_val.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_test.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_train.csv
/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs/y_val.csv


In [2]:
from pathlib import Path

DATA_DIR = Path("/kaggle/input/notebooks/navyatripathi123/02-preprocessing/outputs")

X_train = pd.read_csv(DATA_DIR / "X_train.csv")
X_val = pd.read_csv(DATA_DIR / "X_val.csv")
X_test = pd.read_csv(DATA_DIR / "X_test.csv")

y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_val = pd.read_csv(DATA_DIR / "y_val.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

print(y_train.shape)
print(y_val.shape)
print(y_test.shape)

(28831, 19)
(6178, 19)
(6179, 19)
(28831,)
(6178,)
(6179,)


In [3]:
v2_drop=["emp.var.rate", "euribor3m"]
X_train.drop(columns=v2_drop, inplace=True)
X_val.drop(columns=v2_drop, inplace=True)
X_test.drop(columns=v2_drop, inplace=True)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(28831, 17)
(6178, 17)
(6179, 17)


In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# Identify numerical and categorical columns
cat_cols = X_train.select_dtypes(
    include=['object', 'category']
).columns.tolist()

num_cols = X_train.select_dtypes(
    include=np.number
).columns.tolist()

# Numerical preprocessing
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

# Categorical preprocessing
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(
        handle_unknown='ignore',
        drop='first'
    ))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit on train; transform validation and test
X_train_encoded = preprocessor.fit_transform(X_train)
X_val_encoded = preprocessor.transform(X_val)
X_test_encoded = preprocessor.transform(X_test)

print("Train:", X_train_encoded.shape)
print("Validation:", X_val_encoded.shape)
print("Test:", X_test_encoded.shape)

Train: (28831, 50)
Validation: (6178, 50)
Test: (6179, 50)


In [5]:
from xgboost import XGBClassifier

# xgb = XGBClassifier(
#     n_estimators=200,
#     learning_rate=0.1,
#     max_depth=6,
#     random_state=42,
#     eval_metric='logloss',
#     n_jobs=-1
# )

In [6]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [7]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 50, 200
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.2, log=True
        ),

        "max_depth": trial.suggest_int(
            "max_depth", 3, 7
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 10
        ),

        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),

        "gamma": trial.suggest_float(
            "gamma", 0, 5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha", 0.1, 10,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 10,
            log=True
        )
    }

    model = XGBClassifier(
        **params,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1,
        tree_method="hist"
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc",
        n_jobs=-1
    )

    return scores.mean()

In [8]:
os.makedirs("/kaggle/working/outputs", exist_ok=True)

log_path = "/kaggle/working/outputs/optuna_xgb_trials_06.csv"

def log_optuna_trial(study, trial):

    rows = []

    for t in study.trials:

        row = {
            "Trial": t.number,
            "Current CV ROC-AUC": t.value,
            "Best CV ROC-AUC": study.best_value,
            "Best Trial": study.best_trial.number,
            "Is Best": t.number == study.best_trial.number
        }

        row.update(t.params)
        rows.append(row)

    pd.DataFrame(rows).sort_values("Trial").to_csv(
        log_path,
        index=False
    )

In [9]:
!pip install -q optuna
import optuna

In [10]:
study = optuna.create_study(
    direction="maximize",
    study_name="xgboost_roc_auc"
)

study.optimize(
    objective,
    n_trials=150,
    callbacks=[log_optuna_trial]
)

[I 2026-11-23 07:58:38,164] A new study created in memory with name: xgboost_roc_auc
[I 2026-11-23 07:58:41,836] Trial 0 finished with value: 0.789738314247443 and parameters: {'n_estimators': 135, 'learning_rate': 0.03539517353936233, 'max_depth': 5, 'min_child_weight': 7, 'subsample': 0.8491089215610246, 'colsample_bytree': 0.9101332454068312, 'gamma': 4.889682402724242, 'reg_alpha': 7.289704750424577, 'reg_lambda': 0.021785037410299704}. Best is trial 0 with value: 0.789738314247443.
[I 2026-11-23 07:58:43,468] Trial 1 finished with value: 0.7916965822014994 and parameters: {'n_estimators': 169, 'learning_rate': 0.037217236792663186, 'max_depth': 5, 'min_child_weight': 6, 'subsample': 0.8822703796375946, 'colsample_bytree': 0.9156852765893696, 'gamma': 2.7975477594767995, 'reg_alpha': 0.12913333232928084, 'reg_lambda': 0.0037755356991133783}. Best is trial 1 with value: 0.7916965822014994.
[I 2026-11-23 07:58:44,868] Trial 2 finished with value: 0.7916382264455891 and parameters: {'

In [11]:
optuna_results = pd.read_csv(
    "/kaggle/working/outputs/optuna_xgb_trials_06.csv"
)

optuna_results.round(4)

,Trial,Current CV ROC-AUC,Best CV ROC-AUC,Best Trial,Is Best,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,gamma,reg_alpha,reg_lambda
0,0,0.7897,0.7947,121,False,135,0.0354,5,7,0.8491,0.9101,4.8897,7.2897,0.0218
1,1,0.7917,0.7947,121,False,169,0.0372,5,6,0.8823,0.9157,2.7975,0.1291,0.0038
2,2,0.7916,0.7947,121,False,99,0.0689,4,2,0.6195,0.6685,1.7082,4.6190,0.0025
3,3,0.7904,0.7947,121,False,104,0.0231,5,8,0.7318,0.8782,0.2347,0.7470,0.0012
4,4,0.7920,0.7947,121,False,120,0.0790,4,3,0.7037,0.8916,0.0297,0.9573,0.0040
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,145,0.7935,0.7947,121,False,75,0.0663,7,5,0.9645,0.7705,2.9153,0.1226,0.4894
146,146,0.7935,0.7947,121,False,85,0.0757,7,5,0.9756,0.7511,3.3009,0.1520,0.8446
147,147,0.7937,0.7947,121,False,89,0.0581,7,5,0.9564,0.7737,3.2583,0.1441,0.5843
148,148,0.7936,0.7947,121,False,85,0.0602,7,5,0.9277,0.7513,3.0665,0.1246,0.3598


In [12]:
print("Best Trial:", study.best_trial.number)
print("Best CV ROC-AUC:", study.best_value)
print("Best Parameters:")

for param, value in study.best_params.items():
    print(f"{param}: {value}")

Best Trial: 121
Best CV ROC-AUC: 0.7946645446180733
Best Parameters:
n_estimators: 81
learning_rate: 0.07410532139162289
max_depth: 7
min_child_weight: 5
subsample: 0.9542681887896876
colsample_bytree: 0.7368487523197977
gamma: 3.173131921822334
reg_alpha: 0.1385280525822228
reg_lambda: 0.667197935916209


In [13]:
best_xgb = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric="logloss",
    n_jobs=-1,
    tree_method="hist"
)

best_xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", best_xgb)
])

best_xgb_pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['age', 'campaign', 'pdays',
                                                   'previous', 'cons.price.idx',
                                                   'cons.conf.idx',
                                                   'nr.employed']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ig...
                               gamma=3.173131921822334, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None,
                               learning_rate=0.07410532139162289, max_bin=None,
                               max_cat_threshold=None, max_cat_to_onehot=None,
                               max_delta_step=None, max_depth=7,
                               max_leaves=None, min_child_weight=5, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=81, n_jobs=-1,
                               num_parallel_tree=None, ...))])

In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_val_pred = best_xgb_pipeline.predict(X_val)
y_val_prob = best_xgb_pipeline.predict_proba(X_val)[:, 1]

tuned_xgb_results = {
    "Model": "XGBoost Tuned",
    "Accuracy": accuracy_score(y_val, y_val_pred),
    "Precision": precision_score(y_val, y_val_pred),
    "Recall": recall_score(y_val, y_val_pred),
    "F1": f1_score(y_val, y_val_pred),
    "ROC-AUC": roc_auc_score(y_val, y_val_prob)
}

pd.DataFrame([tuned_xgb_results]).round(4)

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
0,XGBoost Tuned,0.9017,0.6787,0.2428,0.3577,0.8086


In [15]:
# models_cv = {
#     "XGBoost": (xgb, X_train_encoded)
# }

# cv_results = []

# for name, (model, X_train_model) in models_cv.items():

#     scores = cross_validate(
#         model,
#         X_train_model,
#         y_train,
#         cv=cv,
#         scoring=[
#             "roc_auc",
#             "precision",
#             "recall",
#             "f1"
#         ],
#         n_jobs=-1
#     )

#     cv_results.append({
#         "Model": name,
#         "CV ROC-AUC": scores["test_roc_auc"].mean(),
#         "CV Precision": scores["test_precision"].mean(),
#         "CV Recall": scores["test_recall"].mean(),
#         "CV F1": scores["test_f1"].mean()
#     })

# cv_results_df = pd.DataFrame(cv_results)

# cv_results_df.round(4)

In [16]:
# xgb.fit(X_train_encoded, y_train)

In [17]:
# xgb_prob = xgb.predict_proba(X_val_encoded)[:, 1]

In [18]:
# from sklearn.metrics import (
#     accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score,
#     roc_auc_score
# )
# models = {
#     "XGBoost": (xgb, X_val_encoded)
# }

# results = []

# for name, (model, X_val_model) in models.items():

#     y_pred = model.predict(X_val_model)
#     y_prob = model.predict_proba(X_val_model)[:, 1]

#     results.append({
#         "Model": name,
#         "Accuracy": accuracy_score(y_val, y_pred),
#         "Precision": precision_score(y_val, y_pred),
#         "Recall": recall_score(y_val, y_pred),
#         "F1": f1_score(y_val, y_pred),
#         "ROC-AUC": roc_auc_score(y_val, y_prob)
#     })